# 💰 Финансовый помощник: доход, расходы и прогноз накоплений

Notebook-приложение с графическим интерфейсом (`ipywidgets`). Возможности:

- 💾 **Сохранение данных** в файл и загрузка обратно (переживает перезапуск).
- 📥 **Догрузка отчётов**: подгружаешь новый экспорт Clockify — добавляются только новые записи, совпадающие не дублируются.
- ✏️ **Групповое редактирование** записей: выбрать по периоду / дням недели / проекту и поменять ставку, масштабировать часы или удалить.
- ➕ **Добавление** записей о работе вручную.
- 🧾 **Расходы**: ежемесячные (аренда, еда…) и разовые (покупки).
- 🏦 **Текущий баланс счёта**.
- 📈 **Прогноз накоплений**: сколько будет на счету через N недель/месяцев — с доверительным интервалом (Монте-Карло).

> Запусти **все ячейки по порядку** (`Kernel → Restart & Run All`), затем работай с панелями интерфейса. Они появятся прямо под ячейками.

## 0. Установка и импорты

In [2]:
# ipywidgets нужен для интерфейса. Если его нет — установится автоматически.
try:
    import ipywidgets  # noqa
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'ipywidgets'], check=False)

import json, os
from datetime import date, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

plt.rcParams['figure.figsize'] = (11, 4.8)
plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3

WD_NAMES = ['Пн','Вт','Ср','Чт','Пт','Сб','Вс']
WD_OPTIONS = [(WD_NAMES[i], i) for i in range(7)]

# Файлы по умолчанию
CLOCKIFY_CSV = 'Clockify_Time_Report_Detailed_01_04_2026-31_05_2026.csv'  # исходный отчёт
STATE_FILE   = 'finance_state.json'  # сюда сохраняется всё состояние

## 1. Движок (функции)

Здесь вся логика: загрузка отчёта, групповое редактирование, модель часов по дням недели, расходы, прогноз накоплений и сохранение/загрузка состояния. Эти функции используются интерфейсом ниже — отдельно их трогать не нужно.

In [3]:
# ---------- Загрузка отчёта Clockify ----------
def load_clockify(path):
    d = pd.read_csv(path)
    date_only = pd.to_datetime(d['Start Date'], format='%d/%m/%Y')
    # Время начала записи — нужно для надёжной дедупликации при догрузке отчётов
    if 'Start Time' in d.columns:
        start = pd.to_datetime(d['Start Date'].astype(str) + ' ' + d['Start Time'].astype(str),
                               format='%d/%m/%Y %H:%M:%S', errors='coerce').fillna(date_only)
    else:
        start = date_only
    desc = d['Description'] if 'Description' in d.columns else pd.Series(['']*len(d))
    out = pd.DataFrame({
        'date': date_only.dt.normalize(),
        'start': start,
        'project': d['Project'].fillna(''),
        'description': desc.fillna(''),
        'hours': pd.to_numeric(d['Duration (decimal)'], errors='coerce'),
        'rate': pd.to_numeric(d['Billable Rate (USD)'], errors='coerce').fillna(0.0),
    })
    return out.dropna(subset=['hours']).sort_values('start').reset_index(drop=True)

# ---------- Ключ записи и слияние отчётов (без дублей) ----------
def entry_key(df):
    """Уникальный ключ записи: время начала + проект + часы + ставка + описание."""
    s = pd.to_datetime(df['start']).dt.strftime('%Y-%m-%dT%H:%M:%S')
    return (s + '|' + df['project'].astype(str) + '|'
            + df['hours'].round(3).astype(str) + '|'
            + df['rate'].astype(str) + '|' + df['description'].astype(str))

def merge_entries(old, new):
    """Добавляет к old только те записи из new, которых там ещё нет.
    Возвращает (объединённый_df, сколько_добавлено)."""
    if old is None or len(old) == 0:
        return new.sort_values('start').reset_index(drop=True), len(new)
    have = set(entry_key(old))
    is_new = ~entry_key(new).isin(have)
    added = int(is_new.sum())
    combined = pd.concat([old, new[is_new.values]], ignore_index=True)
    combined = (combined.assign(_k=entry_key(combined))
                .drop_duplicates('_k', keep='first').drop(columns='_k')
                .sort_values('start').reset_index(drop=True))
    return combined, added

def detect_rate(df):
    """Текущая ставка = последняя по времени ненулевая."""
    nz = df[df['rate'] > 0].sort_values('date')
    return float(nz['rate'].iloc[-1]) if len(nz) else 0.0

# ---------- Групповое редактирование ----------
def apply_group_edit(df, date_from=None, date_to=None, weekdays=None,
                     project=None, action='set_rate', value=0.0):
    m = pd.Series(True, index=df.index)
    if date_from is not None: m &= df['date'] >= pd.Timestamp(date_from)
    if date_to   is not None: m &= df['date'] <= pd.Timestamp(date_to)
    if weekdays:              m &= df['date'].dt.dayofweek.isin(list(weekdays))
    if project:               m &= df['project'] == project
    n = int(m.sum())
    df = df.copy()
    if   action == 'set_rate':    df.loc[m, 'rate']  = value
    elif action == 'set_hours':   df.loc[m, 'hours'] = value
    elif action == 'scale_hours': df.loc[m, 'hours'] = df.loc[m, 'hours'] * value
    elif action == 'delete':      df = df[~m].reset_index(drop=True)
    return df, n

# ---------- Модель часов по дням недели ----------
def weekday_model(df):
    daily = df.groupby(df['date'].dt.normalize())['hours'].sum().reset_index()
    if len(daily) == 0:
        z = {wd: 0.0 for wd in range(7)}
        return z, {wd: np.array([]) for wd in range(7)}, z
    full = pd.DataFrame({'date': pd.date_range(daily['date'].min(), daily['date'].max(), freq='D')})
    full = full.merge(daily, on='date', how='left'); full['hours'] = full['hours'].fillna(0.0)
    full['wd'] = full['date'].dt.dayofweek; full['worked'] = full['hours'] > 0
    p_work = {wd: float(full.loc[full['wd']==wd, 'worked'].mean() or 0) for wd in range(7)}
    pools  = {wd: full.loc[(full['wd']==wd) & full['worked'], 'hours'].values for wd in range(7)}
    exp_h  = {wd: p_work[wd]*(pools[wd].mean() if len(pools[wd]) else 0.0) for wd in range(7)}
    return p_work, pools, exp_h

# ---------- Расходы ----------
def expand_expenses(expenses, start, end):
    idx = pd.date_range(start, end, freq='D')
    s = pd.Series(0.0, index=idx)
    for e in expenses:
        if e['kind'] == 'monthly':
            for d in idx:
                if d.day == int(e['day']): s[d] += float(e['amount'])
        else:  # once
            t = pd.Timestamp(e['date'])
            if start <= t <= end: s[t] += float(e['amount'])
    return s

# ---------- Прогноз накоплений ----------
def forecast_savings(df, rate, expenses, balance, balance_date, end,
                     n_sims=4000, seed=1):
    p_work, pools, exp_h = weekday_model(df)
    start = pd.Timestamp(balance_date) + pd.Timedelta(days=1)
    end = pd.Timestamp(end)
    if end < start:
        raise ValueError('Дата конца прогноза должна быть позже даты баланса.')
    days = pd.date_range(start, end, freq='D')
    exp = expand_expenses(expenses, start, end)
    rng = np.random.default_rng(seed)
    paths = np.zeros((n_sims, len(days))); run = np.full(n_sims, float(balance))
    mean_run = float(balance); mean_path = []
    for i, d in enumerate(days):
        wd = d.dayofweek
        worked = rng.random(n_sims) < p_work[wd]
        pool = pools[wd]
        h = rng.choice(pool, size=n_sims) if len(pool) else np.zeros(n_sims)
        out = float(exp.iloc[i])
        run = run + worked*h*rate - out
        paths[:, i] = run
        mean_run = mean_run + exp_h[wd]*rate - out
        mean_path.append(mean_run)
    return {
        'days': days, 'mean': np.array(mean_path),
        'p10': np.percentile(paths, 10, axis=0),
        'p90': np.percentile(paths, 90, axis=0),
        'final': paths[:, -1], 'expenses_series': exp,
        'total_expenses': float(exp.sum()),
    }

# ---------- Сохранение / загрузка состояния ----------
def save_state(state, path):
    df = state['entries']
    blob = {
        'entries': [{'date': r['date'].strftime('%Y-%m-%d'),
                     'start': pd.Timestamp(r['start']).strftime('%Y-%m-%dT%H:%M:%S'),
                     'project': str(r['project']),
                     'description': str(r['description']), 'hours': float(r['hours']),
                     'rate': float(r['rate'])} for _, r in df.iterrows()],
        'expenses': state['expenses'],
        'balance': float(state['balance']),
        'balance_date': state['balance_date'].strftime('%Y-%m-%d'),
        'rate': float(state['rate']),
        'currency': state['currency'],
    }
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(blob, f, ensure_ascii=False, indent=2)

def load_state(path):
    with open(path, encoding='utf-8') as f:
        blob = json.load(f)
    e = pd.DataFrame(blob['entries'])
    e['date'] = pd.to_datetime(e['date'])
    # Обратная совместимость: в старых сохранениях нет 'start'
    if 'start' in e.columns:
        e['start'] = pd.to_datetime(e['start']).fillna(e['date'])
    else:
        e['start'] = e['date']
    e['hours'] = pd.to_numeric(e['hours']); e['rate'] = pd.to_numeric(e['rate'])
    return {
        'entries': e.sort_values('start').reset_index(drop=True),
        'expenses': blob.get('expenses', []),
        'balance': float(blob.get('balance', 0.0)),
        'balance_date': pd.to_datetime(blob['balance_date']).date(),
        'rate': float(blob.get('rate', 0.0)),
        'currency': blob.get('currency', 'USD'),
    }
print('Движок загружен ✓')

Движок загружен ✓


## 2. Инициализация

Если рядом есть сохранённый файл состояния — грузим его. Иначе берём исходный отчёт Clockify.

In [4]:
if os.path.exists(STATE_FILE):
    STATE = load_state(STATE_FILE)
    print(f'Загружено состояние из {STATE_FILE}')
else:
    _df = load_clockify(CLOCKIFY_CSV)
    STATE = {
        'entries': _df,
        'expenses': [],
        'balance': 0.0,
        'balance_date': _df['date'].max().date() if len(_df) else date.today(),
        'rate': detect_rate(_df),
        'currency': 'USD',
    }
    print(f'Импортирован отчёт {CLOCKIFY_CSV}: {len(_df)} записей, ставка {STATE["rate"]:.0f}')

print('Записей о работе:', len(STATE['entries']), '| расходов:', len(STATE['expenses']),
      '| баланс:', STATE['balance'], STATE['currency'])

Импортирован отчёт Clockify_Time_Report_Detailed_01_04_2026-31_05_2026.csv: 61 записей, ставка 800
Записей о работе: 61 | расходов: 0 | баланс: 0.0 USD


## 3. Интерфейс

Ниже — приложение с вкладками: **Данные · Работа · Расходы · Прогноз**. Все изменения держатся в памяти; чтобы сохранить на диск — вкладка «Данные» → «💾 Сохранить».

In [ ]:
# ============================ ИНТЕРФЕЙС ============================
out_status = widgets.Output()
out_table  = widgets.Output()
out_exp    = widgets.Output()
out_fc     = widgets.Output()

def notify(msg):
    with out_status:
        clear_output(wait=True); print(msg)

# ---------- общие виджеты состояния ----------
w_balance  = widgets.FloatText(value=STATE['balance'], description='Баланс:')
w_baldate  = widgets.DatePicker(value=STATE['balance_date'], description='на дату:')
w_rate     = widgets.FloatText(value=STATE['rate'], description='Ставка/ч:')
w_currency = widgets.Text(value=STATE['currency'], description='Валюта:')

def _sync_state(_=None):
    STATE['balance']  = w_balance.value
    STATE['balance_date'] = w_baldate.value or STATE['balance_date']
    STATE['rate']     = w_rate.value
    STATE['currency'] = w_currency.value
for w in (w_balance, w_baldate, w_rate, w_currency):
    w.observe(_sync_state, names='value')

def sync_widgets_from_state():
    w_balance.value  = STATE['balance']
    w_baldate.value  = STATE['balance_date']
    w_rate.value     = STATE['rate']
    w_currency.value = STATE['currency']

# ================= Вкладка: ДАННЫЕ (save/load/import) =================
w_statefile = widgets.Text(value=STATE_FILE, description='Файл:')
w_csvfile   = widgets.Text(value=CLOCKIFY_CSV, description='CSV:')
b_save = widgets.Button(description='💾 Сохранить', button_style='success')
b_load = widgets.Button(description='📂 Загрузить', button_style='info')
b_imp  = widgets.Button(description='📥 Импорт (заменить)')
b_merge= widgets.Button(description='➕ Догрузить (без дублей)', button_style='success')

def on_save(_):
    _sync_state(); save_state(STATE, w_statefile.value); notify(f'Сохранено в {w_statefile.value}')
def on_load(_):
    global STATE
    try:
        STATE = load_state(w_statefile.value); sync_widgets_from_state()
        refresh_table(); refresh_expenses(); notify(f'Загружено из {w_statefile.value}')
    except Exception as e:
        notify(f'Ошибка загрузки: {e}')
def on_import(_):
    try:
        STATE['entries'] = load_clockify(w_csvfile.value)
        STATE['rate'] = detect_rate(STATE['entries']); w_rate.value = STATE['rate']
        refresh_table(); notify(f'Импортировано {len(STATE["entries"])} записей (старые заменены)')
    except Exception as e:
        notify(f'Ошибка импорта: {e}')
def on_merge(_):
    try:
        new = load_clockify(w_csvfile.value)
        STATE['entries'], added = merge_entries(STATE['entries'], new)
        if not STATE['rate']:
            STATE['rate'] = detect_rate(STATE['entries']); w_rate.value = STATE['rate']
        refresh_table()
        notify(f'Догружено: +{added} новых записей (дубликаты пропущены). '
               f'Всего стало: {len(STATE["entries"])}')
    except Exception as e:
        notify(f'Ошибка догрузки: {e}')
b_save.on_click(on_save); b_load.on_click(on_load)
b_imp.on_click(on_import); b_merge.on_click(on_merge)

tab_data = widgets.VBox([
    widgets.HTML('<b>Сохранение / загрузка состояния</b>'),
    widgets.HBox([w_statefile, b_save, b_load]),
    widgets.HTML('<b>Отчёты Clockify</b>'),
    widgets.HBox([w_csvfile]),
    widgets.HBox([b_imp, b_merge]),
    widgets.HTML('<small>«Импорт» <b>заменяет</b> все записи. «Догрузить» <b>добавляет</b> только новые '
                 'записи из отчёта — совпадающие пропускаются.</small>'),
    widgets.HTML('<b>Параметры</b>'),
    widgets.HBox([w_balance, w_baldate]),
    widgets.HBox([w_rate, w_currency]),
])

# ================= Вкладка: РАБОТА (table + group edit + add) =================
def refresh_table():
    df = STATE['entries']
    with out_table:
        clear_output(wait=True)
        if len(df) == 0:
            print('Нет записей.'); return
        cur = STATE['currency']
        tot_h = df['hours'].sum(); tot_a = (df['hours']*df['rate']).sum()
        view = df.copy()
        view['amount'] = view['hours']*view['rate']
        view = view.sort_values('date', ascending=False).head(15)
        view['date'] = view['date'].dt.strftime('%d.%m.%Y')
        view = view[['date','project','hours','rate','amount']]
        view.columns = ['Дата','Проект','Часы','Ставка','Сумма']
        display(HTML(f'<b>Всего:</b> {len(df)} записей, {tot_h:.1f} ч, '
                     f'{tot_a:,.0f} {cur} &nbsp;|&nbsp; <i>последние 15:</i>'))
        display(view.style.format({'Часы':'{:.2f}','Ставка':'{:.0f}','Сумма':'{:,.0f}'})
                .hide(axis='index'))

# --- групповое редактирование ---
ge_from = widgets.DatePicker(description='с даты:')
ge_to   = widgets.DatePicker(description='по дату:')
ge_wd   = widgets.SelectMultiple(options=WD_OPTIONS, description='Дни нед.:',
                                 rows=4, layout=widgets.Layout(width='160px'))
ge_proj = widgets.Text(description='Проект:', placeholder='пусто = любой')
ge_act  = widgets.Dropdown(options=[('Установить ставку','set_rate'),
                                    ('Установить часы','set_hours'),
                                    ('Умножить часы на','scale_hours'),
                                    ('Удалить записи','delete')], description='Действие:')
ge_val  = widgets.FloatText(value=0.0, description='Значение:')
b_preview = widgets.Button(description='👁 Предпросмотр')
b_apply   = widgets.Button(description='✔ Применить', button_style='warning')

def _ge_args():
    return dict(date_from=ge_from.value, date_to=ge_to.value,
                weekdays=tuple(ge_wd.value), project=(ge_proj.value or None))
def on_preview(_):
    _, n = apply_group_edit(STATE['entries'], action='delete', **_ge_args())
    notify(f'Под фильтр попадает записей: {n}')
def on_apply(_):
    df2, n = apply_group_edit(STATE['entries'], action=ge_act.value,
                              value=ge_val.value, **_ge_args())
    STATE['entries'] = df2
    if ge_act.value in ('set_rate',): STATE['rate'] = detect_rate(df2); w_rate.value = STATE['rate']
    refresh_table(); notify(f'Изменено записей: {n} (действие: {ge_act.label})')
b_preview.on_click(on_preview); b_apply.on_click(on_apply)

# --- добавить запись ---
ad_date = widgets.DatePicker(description='Дата:', value=date.today())
ad_proj = widgets.Text(description='Проект:')
ad_hours= widgets.FloatText(description='Часы:', value=0.0)
ad_rate = widgets.FloatText(description='Ставка:', value=STATE['rate'])
b_add   = widgets.Button(description='➕ Добавить', button_style='success')
def on_add(_):
    if not ad_date.value: notify('Укажи дату.'); return
    row = {'date': pd.Timestamp(ad_date.value).normalize(),
           'start': pd.Timestamp(ad_date.value), 'project': ad_proj.value,
           'description': '', 'hours': ad_hours.value, 'rate': ad_rate.value}
    STATE['entries'] = pd.concat([STATE['entries'], pd.DataFrame([row])],
                                 ignore_index=True).sort_values('start').reset_index(drop=True)
    refresh_table(); notify('Запись добавлена.')
b_add.on_click(on_add)

tab_work = widgets.VBox([
    out_table,
    widgets.HTML('<hr><b>Групповое изменение</b> (фильтр → действие)'),
    widgets.HBox([ge_from, ge_to, ge_wd, ge_proj]),
    widgets.HBox([ge_act, ge_val, b_preview, b_apply]),
    widgets.HTML('<hr><b>Добавить запись</b>'),
    widgets.HBox([ad_date, ad_proj, ad_hours, ad_rate, b_add]),
])

# ================= Вкладка: РАСХОДЫ =================
ex_kind = widgets.ToggleButtons(options=[('Ежемесячный','monthly'),('Разовый','once')],
                                description='Тип:')
ex_name = widgets.Text(description='Название:')
ex_amt  = widgets.FloatText(description='Сумма:', value=0.0)
ex_day  = widgets.IntSlider(description='День мес.:', value=1, min=1, max=28)
ex_date = widgets.DatePicker(description='Дата:', value=date.today())
b_exadd = widgets.Button(description='➕ Добавить расход', button_style='success')
ex_delidx = widgets.IntText(description='№ удалить:', value=0)
b_exdel = widgets.Button(description='🗑 Удалить', button_style='danger')

def refresh_expenses():
    with out_exp:
        clear_output(wait=True)
        ex = STATE['expenses']; cur = STATE['currency']
        if not ex: print('Расходов пока нет.'); return
        rows = []
        monthly_total = 0.0
        for i, e in enumerate(ex):
            if e['kind'] == 'monthly':
                rows.append((i, 'ежемес.', e['name'], f"{e['amount']:,.0f} {cur}", f"{e['day']} число"))
                monthly_total += float(e['amount'])
            else:
                rows.append((i, 'разовый', e['name'], f"{e['amount']:,.0f} {cur}", e['date']))
        t = pd.DataFrame(rows, columns=['№','Тип','Название','Сумма','Когда'])
        display(t.style.hide(axis='index'))
        print(f'Ежемесячных расходов суммарно: {monthly_total:,.0f} {cur}/мес')

def on_exadd(_):
    if ex_kind.value == 'monthly':
        e = {'kind':'monthly','name':ex_name.value or 'Расход','amount':ex_amt.value,'day':int(ex_day.value)}
    else:
        if not ex_date.value: notify('Укажи дату разового расхода.'); return
        e = {'kind':'once','name':ex_name.value or 'Расход','amount':ex_amt.value,
             'date':ex_date.value.strftime('%Y-%m-%d')}
    STATE['expenses'].append(e); refresh_expenses(); notify('Расход добавлен.')
def on_exdel(_):
    i = ex_delidx.value
    if 0 <= i < len(STATE['expenses']):
        rem = STATE['expenses'].pop(i); refresh_expenses(); notify(f'Удалён: {rem["name"]}')
    else:
        notify('Неверный номер.')
b_exadd.on_click(on_exadd); b_exdel.on_click(on_exdel)

tab_exp = widgets.VBox([
    out_exp,
    widgets.HTML('<hr><b>Добавить расход</b> (для «ежемесячного» — день месяца, для «разового» — дата)'),
    ex_kind,
    widgets.HBox([ex_name, ex_amt]),
    widgets.HBox([ex_day, ex_date]),
    b_exadd,
    widgets.HTML('<hr>'),
    widgets.HBox([ex_delidx, b_exdel]),
])

# ================= Вкладка: ПРОГНОЗ =================
fc_end = widgets.DatePicker(description='Прогноз до:',
                            value=(STATE['balance_date'] + pd.Timedelta(days=90)))
b_fc   = widgets.Button(description='📈 Построить прогноз', button_style='primary')

def on_forecast(_):
    _sync_state()
    with out_fc:
        clear_output(wait=True)
        try:
            r = forecast_savings(STATE['entries'], STATE['rate'], STATE['expenses'],
                                 STATE['balance'], STATE['balance_date'], fc_end.value)
        except Exception as e:
            print('Ошибка:', e); return
        cur = STATE['currency']; days = r['days']
        fig, ax = plt.subplots(figsize=(11, 5))
        ax.axhline(STATE['balance'], color='gray', ls=':', lw=1, label='Старт. баланс')
        ax.plot(days, r['mean'], color='#2e7d32', lw=2, label='Ожидаемый баланс')
        ax.fill_between(days, r['p10'], r['p90'], color='#2e7d32', alpha=0.18, label='80% интервал')
        ex_days = r['expenses_series'][r['expenses_series'] > 0]
        if len(ex_days):
            ax.scatter(ex_days.index, np.interp(mdates.date2num(ex_days.index),
                       mdates.date2num(days), r['mean']), color='#c62828', s=25, zorder=5,
                       label='Расходы')
        ax.set_title('Прогноз накоплений'); ax.set_ylabel(cur)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m')); ax.legend()
        plt.tight_layout(); plt.show()
        m = r['final'].mean()
        print(f'На {pd.Timestamp(fc_end.value).date()}:')
        print(f'  Ожидаемый баланс:  {m:,.0f} {cur}')
        print(f'  Интервал 80%:      {np.percentile(r["final"],10):,.0f} — {np.percentile(r["final"],90):,.0f} {cur}')
        print(f'  Прирост за период: {m-STATE["balance"]:,.0f} {cur}')
        print(f'  Из них расходы:    -{r["total_expenses"]:,.0f} {cur}')
b_fc.on_click(on_forecast)

tab_fc = widgets.VBox([
    widgets.HTML('Прогноз строится от <b>баланса на дату</b> (вкладка «Данные») и вперёд до выбранной даты.'),
    widgets.HBox([fc_end, b_fc]),
    out_fc,
])

# ================= Сборка вкладок =================
tabs = widgets.Tab(children=[tab_data, tab_work, tab_exp, tab_fc])
for i, t in enumerate(['Данные','Работа','Расходы','Прогноз']):
    tabs.set_title(i, t)

refresh_table(); refresh_expenses()
display(tabs, out_status)

## 4. Быстрый обзор (статичные графики)

Необязательная секция: графики по текущим данным `STATE` без интерфейса. Перезапусти ячейку после изменений, чтобы обновить.

In [ ]:
df = STATE['entries']; cur = STATE['currency']
daily = df.groupby(df['date'].dt.normalize()).agg(
    hours=('hours','sum'),
    amount=('hours', lambda h: 0)).reset_index().rename(columns={'date':'date'})
daily['amount'] = df.assign(a=df['hours']*df['rate']).groupby(df['date'].dt.normalize())['a'].sum().values

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(daily['date'], daily['amount'], width=0.9, color='#2e7d32', alpha=0.85)
axes[0].set_title('Доход по дням'); axes[0].set_ylabel(cur)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
axes[1].plot(daily['date'], daily['amount'].cumsum(), color='#1565c0', lw=2)
axes[1].fill_between(daily['date'], daily['amount'].cumsum(), alpha=0.2, color='#1565c0')
axes[1].set_title('Накопительный доход'); axes[1].set_ylabel(cur)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
plt.tight_layout(); plt.show()

by_wd = daily.assign(wd=daily['date'].dt.dayofweek).groupby('wd')['amount'].mean().reindex(range(7))
plt.figure(figsize=(8,3.5))
plt.bar([WD_NAMES[i] for i in range(7)], by_wd.values, color='#ef6c00', alpha=0.85)
plt.title('Средний доход за рабочий день недели'); plt.ylabel(cur); plt.tight_layout(); plt.show()

## 5. Как пользоваться

1. **Запусти всё** (`Restart & Run All`) — под разделом 3 появятся вкладки.
2. **Данные** → задай текущий **баланс** и **дату**, при желании поменяй **ставку**/валюту. Здесь же кнопки **Сохранить / Загрузить**, а также работа с отчётами Clockify:
   - **📥 Импорт (заменить)** — полностью перезаписывает записи новым отчётом;
   - **➕ Догрузить (без дублей)** — добавляет к уже сохранённым данным только новые записи из отчёта; те, что уже есть, пропускаются. Это то, что нужно, когда выгружаешь свежий отчёт за новый период.
3. **Работа** → смотри записи, меняй их **группами** (например: «с 01.04 по 29.04, действие — Установить ставку, значение 800») или добавляй вручную.
4. **Расходы** → добавь ежемесячные (аренда, еда) и разовые (крупные покупки).
5. **Прогноз** → выбери дату и нажми «Построить прогноз»: получишь график роста баланса с интервалом 80% и итоговые цифры.
6. Не забудь **💾 Сохранить** — иначе после перезапуска ядра данные вернутся к исходному CSV.

**Логика прогноза:** будущий доход моделируется по твоему реальному режиму (вероятность работы и часы для каждого дня недели) × ставка, из него вычитаются расходы по датам, и всё это накручивается на стартовый баланс. Интервал 80% показывает разброс из-за неравномерной загрузки — ориентируйся на него, а не только на среднее.